# Introducció a la cerca

Aquest *Jupyter Notebook* presenta un **problema simple**, però realista, que es pot resoldre tant amb **cerca exhaustiva** com amb **cerca local heurística**.

Aquí implementarem una solució a un problema de cerca i compararem *dues aproximacions de cerca diferents* aplicades al mateix problema.


------------------------------------------------------------------------

## 1. Problema: assignació de torns a treballadors

### 1.1 Context

Una empresa petita ha d’organitzar els torns de la setmana segons les següents dades:

-   **6 treballadors**: A, B, C, D, E, F
-   **3 torns**: Matí, Tarda, Nit
-   **2 persones per torn**

Cada treballador té una **preferència** per treballar a cada torn.

L'**objectiu** és assignar cada treballador a un torn, respectant la capacitat (2 persones per torn), i *minimitzant el cost total*.


------------------------------------------------------------------------

## 2. Modelització del problema

### 2.1 Treballadors, torns i preferències

Codifiquem tot en una única estructura de dades que podem fàcilment manipular, un diccionari:


In [5]:
import copy

problema = {'treballadors': ['A', 'B', 'C', 'D', 'E', 'F'],
            'torns': ['Matí', 'Tarda', 'Nit'],
            'capacitat': 2,
            "costos": {
                'A': {'Matí': 1, 'Tarda': 4, 'Nit': 5},
                'B': {'Matí': 2, 'Tarda': 3, 'Nit': 4},
                'C': {'Matí': 5, 'Tarda': 1, 'Nit': 3},
                'D': {'Matí': 4, 'Tarda': 2, 'Nit': 1},
                'E': {'Matí': 3, 'Tarda': 5, 'Nit': 2},
                'F': {'Matí': 4, 'Tarda': 1, 'Nit': 3},
            }}

------------------------------------------------------------------------

### 2.2 Representació d’un estat

L'estat representa una assignació (parcial) de treballadors a torns

Exemple:

- A: Matí
- B: Nit

Ens podem guardar l'assignació de persones a torns, així com podem precalcular un compteig de persones assignades per torn:


In [7]:
estat = {
    'assignacio': {'A': 'Matí', 'B': 'Nit'},
    'ocupacio': {"Matí": 1, "Tarda": 0, "Nit": 1}
}



------------------------------------------------------------------------

## 3. Cerca exhaustiva

### 3.1 Cerca de cost uniforme (UCS)

Farem una **cerca de cost uniforme**:

-   Cada node és una assignació parcial (un treballador a un torn)
-   El cost acumulat és la suma de preferències
-   S’expandeix sempre l’estat amb *menor cost acumulat*
-   El primer estat complet trobat és l'**òptim**

------------------------------------------------------------------------

### 3.2 Estat inicial

Inicialment, tot estarà buit:

In [ ]:
estat_inicial = {
    'assignacio': {},
    'ocupacio': {t: 0 for t in problema['torns']}
}

i podrem definir, si ens assegurem que no creem cap estat que violi la restricció de capacitat, que un estat serà complet si el nombre d'assignacions sigui igual al nombre de treballadors:

In [8]:
def estat_complet(estat, problema):
    return len(estat['assignacio']) == len(problema['treballadors'])


------------------------------------------------------------------------

### 3.3 Generació de successors

Si assumim que a cada nivell assignem un treballador a algun torn, tindrem que els successors d'un estat actual són tots els estats que considerin assignar el següent treballador a cada torn:

In [15]:
import copy

def successors(estat, problema):
    # ja estan tots assignats?
    if estat_complet(estat, problema):
        return []

    assign = estat['assignacio']
    ocup = estat['ocupacio']


    seg_treballador = problema['treballadors'][len(assign)]
    succs = []

    for s in problema['torns']:
        if ocup[s] < problema['capacitat']: # ja està el torn complet?

             # creem un nou possible estat com a còpia de l'anterior i afegir la nova assignació
            estat_nou = copy.deepcopy(estat)
            estat_nou['assignacio'][seg_treballador] = s
            estat_nou['ocupacio'][s] += 1

            succs.append(( estat_nou,
                           problema['costos'][seg_treballador][s]) # cost associat
            )

    return succs


------------------------------------------------------------------------

### 3.4 Algorisme de cerca de cost uniforme

Per implementar la cerca de cost uniforme, només necessitem mantenir una cua de prioritat (que implementarem amb `heapq`) i prioritzar el mínim cost:


In [23]:
import heapq
import itertools
counter = itertools.count()

def uniform_cost_search(estat_inicial, problema):
    frontera = []
    heapq.heappush(frontera,
                   (0, next(counter), estat_inicial))

    while frontera: # mentre no estigui buida la frontera
        cost_actual, _, estat_actual = heapq.heappop(frontera)

        # Si és un estat complet, ja hem acabat!
        if estat_complet(estat_actual, problema):
            return estat_actual, cost_actual

        # Si no, expandim el node (estat) actual
        for succ, cost in successors(estat_actual, problema):
            heapq.heappush(frontera, (
                cost_actual + cost, next(counter), succ ))

    return None

Fixa't que la definició del mètode és agnòstica en el sentit que no inclou informació del problema!

------------------------------------------------------------------------

### 3.5 Execució

Ara ja podem executar l'algoritme per trobar la solució:


In [24]:
millor_assign, millor_cost = uniform_cost_search(estat_inicial, problema)
millor_assign['assignacio'], millor_cost

({'A': 'Matí',
  'B': 'Matí',
  'C': 'Tarda',
  'D': 'Nit',
  'E': 'Nit',
  'F': 'Tarda'},
 8)


------------------------------------------------------------------------

## 4. Cerca local

Ara resolguem el **mateix problema** amb cerca local.

L'aproximació és una mica diferent, perquè considerem en tot cas sempre estats complets, i ens movem per l'espai d'estats complets buscant el millor.

------------------------------------------------------------------------

### 4.1 Estat complet

Un estat complet és una assignació de treballadors a torns vàlida (amb 2 persones per torn). Així, assumint un ordre fixe entre els treballadors, podem representar un estat com un vector de torns en què el primer torn és el que s'assigna al primer treballador, etc.

Així, la solució que ha trobat anteriorment l'algoritme UCS seria:


In [30]:
estat = ['Matí', 'Matí', 'Tarda', 'Nit', 'Nit', 'Tarda']

Podem crear un estat aleatori si simplement assignment cada torn a un treballador escollit de manera aleatòria:

In [39]:
import random

def assignacio_inicial_aleatoria(problema):
    treb_aleatoris = list(range(len(problema['treballadors'])))
    random.shuffle(treb_aleatoris)
    assign = ['']*len(treb_aleatoris)
    idx = 0
    for torn in problema['torns']:
        for _ in range(problema['capacitat']):
            assign[treb_aleatoris[idx]] = torn
            idx += 1
    return assign


------------------------------------------------------------------------

### 4.2 Veïnat

Amb aquesta nova representació dels estats, la creació dels veïns (estats que podem crear amb modificacions mínimes-locals) es pot fer amb l'intercanvi del torn de dos treballadors. Així ens assegurem que **tots els veïns** són també **vàlids**.

El veïnat està composat pel conjunt de tots els intercanvis de torn possibles:


In [32]:
def veinat(estat):
    veins = []

    for i in range(len(estat)):
        for j in range(i+1, len(estat)):
            if estat[i] != estat[j]: #  no té sentit canviar el torn amb qui té el mateix torn!
                estat_nou = estat.copy()
                estat_nou[i], estat_nou[j] = estat[j], estat[i]
                veins.append(estat_nou)

    return veins


------------------------------------------------------------------------

### 4.3 Càlcul del cost

Per calcular el cost total d'un estat ara simplement hem de sumar el cost de totes les assignacions:

In [33]:
def calcular_cost(estat, problema):
    return sum(problema['costos'][treb][torn] for treb,torn in zip(problema['treballadors'],estat))


------------------------------------------------------------------------

### 4.4 Algoritme de cerca local

Un algoritme de cerca local que, per exemple, es mou cap al primer veí que millora l'estat actual, seria simplement:


In [34]:
def cerca_local(problema):
    estat_actual = assignacio_inicial_aleatoria(problema)
    cost_actual = calcular_cost(estat_actual, problema)

    improved = True
    while improved:
        veins = veinat(estat_actual)
        # random.shuffle(veins) # podriem reorganitzar aleatoriament el conjunt de veïns
                                # per no dependre de l'ordre de la seva creació
        improved = False

        for vei in veins:
            c = calcular_cost(vei, problema)
            if c < cost_actual:
                estat_actual = vei
                cost_actual = c
                improved = True
                break

    return estat_actual, cost_actual


------------------------------------------------------------------------

### 5.5 Execucions

Ara ja podem executar la cerca local:

In [40]:
millor_estat, millor_cost = cerca_local(problema)

{treb:torn for treb, torn in zip(problema['treballadors'], millor_estat)}, millor_cost

({'A': 'Matí',
  'B': 'Matí',
  'C': 'Tarda',
  'D': 'Nit',
  'E': 'Nit',
  'F': 'Tarda'},
 8)


------------------------------------------------------------------------

## 5. Conclusions

Amb aquest notebook, hem vist:

-   com un problema realista pot formular-se com un problema de cerca
-   la diferència entre **cerca exhaustiva** i **cerca local**, també en la formulació de l'estat
-   la cerca local com solució ràpida alternativa, tot i no garantir l’òptim com sí fa l'exhaustiva
